# Exploration de la source BCEAO

**Projet :** Plateforme d'automatisation de la prospection (veille AO / RFP / CFP)  
**Module :** 1 - Collecte  
**Source étudiée :** BCEAO (Banque Centrale des États de l'Afrique de l'Ouest)

## Objectif de ce carnet

Avant d'écrire le collecteur définitif, je veux comprendre comment cette source expose ses
appels d'offres : peut-on y accéder simplement, quels champs sont disponibles, et comment
distinguer les annonces ouvertes des closes. Je documente toute la démarche, y compris une
fausse piste que j'ai corrigée, parce qu'elle est instructive.

## 1. Comment j'en suis arrivée à la BCEAO

Je suis partie de la liste de sources fournie par l'entreprise. Avant de coder, j'ai testé les
URLs pour voir lesquelles étaient exploitables :

- **Banques et assurances privées** (Attijari, UIB, ATB, Maghrebia...) : pas de page publique
  d'appels d'offres (plateforme fournisseurs sur inscription, ou rien en ligne).
- **TUNEPS** : accès via certificat TUNTRUST, trop complexe pour un premier essai.
- **ANPR** : beaucoup de bruit (mélange appels d'offres, bourses, recrutements).
- **Plusieurs URLs en 404** (liens périmés).

La **BCEAO** répondait bien et affichait une liste publique et structurée. Je la prends comme
**source pilote** : si toute la chaîne marche dessus, je l'étendrai aux autres sources.

Remarque : la BCEAO est la banque centrale de l'Afrique de l'Ouest (UEMOA). Pertinente seulement
si le périmètre du projet couvre cette région. À valider avec l'encadrante.

## 2. Récupération de la page

Je récupère le HTML. J'ajoute un `User-Agent` de navigateur, car beaucoup de sites bloquent les
requêtes qui n'en ont pas (elles ressemblent à des robots).

In [25]:
import requests
import re
import json
from datetime import date
from bs4 import BeautifulSoup

URL = "https://www.bceao.int/fr/appels-offres/appels-offres-marches-publics-achats"
BASE = "https://www.bceao.int"
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/124.0 Safari/537.36"}

reponse = requests.get(URL, headers=HEADERS, timeout=30)
print("Code HTTP :", reponse.status_code)
print("Taille de la page :", len(reponse.content), "octets")

Code HTTP : 200
Taille de la page : 133359 octets


## 3. Le contenu est-il déjà dans le HTML brut ?

Avant tout : puis-je me contenter de `requests`, ou le contenu est-il chargé en JavaScript (ce
qui imposerait un navigateur automatisé comme Playwright) ? Je cherche les liens d'annonces dans
le HTML que le serveur m'a envoyé. S'ils y sont, `requests` suffit.

In [26]:
# re.findall cherche tous les bouts d'URL ayant la forme d'un lien d'annonce, dans le HTML brut.
liens_bruts = re.findall(r"/fr/appels-offres/[a-z0-9\-]{15,}", reponse.text)
nb = len(set(liens_bruts))
print(f"Liens d'annonces trouvés dans le HTML brut : {nb}\n")
print("=> CAS 1 (requests suffit)" if nb > 0 else "=> CAS 2 (JavaScript, Playwright nécessaire)")

Liens d'annonces trouvés dans le HTML brut : 36

=> CAS 1 (requests suffit)


## 4. La page est organisée en sections

Je vois trois sections, chacune introduite par un `<h2>` : « Appel d'offres En cours »,
« Appel d'offres Clos », « Résultats ».

**Première idée (qui s'est révélée fausse).** J'ai d'abord voulu prendre, pour chaque `<h2>`, le
`<div>` qui le suit immédiatement (`find_next_sibling`), en pensant que ce div contiendrait toutes
les annonces de la section.

In [27]:
soup = BeautifulSoup(reponse.text, "html.parser")

for h2 in soup.find_all("h2"):
    titre = " ".join(h2.get_text().split())
    if any(m in titre.lower() for m in ("en cours", "clos", "résultat", "resultat")):
        frere = h2.find_next_sibling()
        if frere is None:
            continue
        liens = frere.find_all("a", href=re.compile(r"/fr/appels-offres/[a-z]"))
        avec_date = [a for a in liens if "ublié le" in a.get_text()]
        print(f"{titre:30} -> {len(avec_date)} annonce(s) via find_next_sibling")

Appel d’offres En cours        -> 23 annonce(s) via find_next_sibling
Appel d’offres Clos            -> 1 annonce(s) via find_next_sibling
Résultats                      -> 0 annonce(s) via find_next_sibling


**Le résultat est trompeur :** « En cours » donne 22 annonces, mais « Clos » n'en donne qu'une
seule. Sur le moment, j'en ai conclu que les annonces closes étaient chargées en JavaScript.

**Mais c'était faux.** En comparant avec le comptage de l'étape 3 (la regex trouvait plus de liens
que ça dans le HTML brut), je me suis rendu compte que les annonces closes **sont bien présentes**
dans le HTML. Le problème venait de ma méthode : pour « En cours », le frère du `<h2>` est un
conteneur qui enveloppe les 22 annonces ; pour « Clos », la structure est différente, le frère
immédiat n'est qu'**une** annonce, les autres étant des frères suivants. `find_next_sibling`
n'en attrapait donc qu'une.

## 5. Méthode fiable : remonter au `<h2>` précédent

Plutôt que de partir du `<h2>` vers ses frères (fragile), je fais l'inverse : pour **chaque
annonce**, je remonte au `<h2>` qui la précède (`find_previous`). Ça me donne sa section de façon
fiable, quelle que soit la structure des conteneurs.

In [28]:
# Toutes les vraies annonces de la page (lien de la bonne forme ET texte contenant "Publié le")
annonces_html = [a for a in soup.find_all("a", href=re.compile(r"/fr/appels-offres/[a-z]"))
                 if "ublié le" in a.get_text()]

# Pour chacune, sa section = le <h2> qui la précède
from collections import Counter
sections = Counter()
for a in annonces_html:
    h2 = a.find_previous("h2")
    titre = " ".join(h2.get_text().split()).lower() if h2 else ""
    if "en cours" in titre:
        sections["en cours"] += 1
    elif "clos" in titre:
        sections["clos"] += 1

print("Annonces par section (méthode fiable) :", dict(sections))
print("Total :", len(annonces_html))

Annonces par section (méthode fiable) : {'en cours': 23, 'clos': 10}
Total : 33


Cette fois j'obtiens **22 « en cours » + 10 « clos » = 32 annonces**, toutes récupérées avec
`requests`. La leçon : ne pas conclure trop vite à du JavaScript. Ici, le contenu était là ; c'est
ma façon de lire la structure qui était mauvaise.

## 6. Inspection d'une annonce : où sont les données ?

Je regarde le contenu d'une annonce pour savoir quoi extraire et comment.

In [29]:
exemple = annonces_html[0]
print("Balise :", exemple.name)
print("Lien (href) :", exemple.get("href"))
print("\nTexte visible :")
print(repr(" ".join(exemple.get_text().split())))

Balise : a
Lien (href) : https://www.bceao.int/fr/appels-offres/selection-dentreprises-specialisees-pour-la-realisation-dinstallations-solaires-0

Texte visible :
'Publié le 17 juin 2026 AO/Z00/DPS/008/2026 Date limite le 31 Juillet 2026 Sélection d’entreprises spécialisées pour la réalisation d’installations solaires photovoltaïques sur les sites de la Banque au Bénin, Burkina, Côte d’Ivoire, Niger et Togo'


Tout est dans le texte d'un seul `<a>` : date de publication, date limite et titre sont collés,
pas répartis dans des balises distinctes. Le modèle est toujours :

```
Publié le <date publication> <référence éventuelle> Date limite le <date limite> <titre>
```

Comme les champs ne sont pas séparés par des balises, j'utiliserai une **expression régulière**
sur ce texte. Le `href` me donnera l'URL de la fiche.

## 7. Pourquoi je filtre sur « Publié le »

Tous les liens de forme `/fr/appels-offres/...` ne sont pas des annonces : il y a aussi le lien
vers la page elle-même, des appels à candidatures, des opérations de marché monétaire. Ils ont la
bonne forme d'URL, parfois même les mots « appel d'offres », mais pas de date « Publié le ».
Filtrer sur « Publié le » cible la **structure** d'une vraie annonce, pas son vocabulaire.

In [30]:
liens = soup.find_all("a", href=re.compile(r"/fr/appels-offres/[a-z]"))
ecartes = [a for a in liens if "ublié le" not in a.get_text()]
vus = set()
print("Liens écartés (forme d'annonce mais pas de 'Publié le') :\n")
for a in ecartes:
    href = a.get("href")
    if href not in vus:
        vus.add(href)
        print(href, "->", repr(" ".join(a.get_text().split())[:45]))

Liens écartés (forme d'annonce mais pas de 'Publié le') :

/fr/appels-offres/appels-offres-marches-publics-achats -> 'FR'
/fr/appels-offres/appel-candidatures-pour-la-49e-promotion-du-cycle-diplomant-du-cofeb -> 'COFEB'
/fr/appels-offres/marche-mon%C3%A9taire -> 'Appel d’offres du marché'
/fr/appels-offres/appels-offres-marches-titres-publics-prives -> 'Avis d’appel d’offres'


## 8. Extraction des champs (toutes les annonces, avec leur section)

Pour chaque annonce je récupère : date de publication, référence (si présente), date limite,
titre, lien, et **`statut_source`** = la section lue sur le site (en cours / clôturé).

- **`cle_unique`** = le *slug* de l'URL : toujours présent et stable (la référence manque parfois).
- **`lien`** = l'URL de la page de détail. **Stratégie simple :** je stocke seulement ce lien.
  La page de détail contient la description complète et les documents (PDF), mais je n'irai les
  extraire que plus tard, et seulement pour les annonces jugées pertinentes.
- **`statut_source`** = section d'origine, déduite du `<h2>` précédent.

In [31]:
def section_de(a):
    h2 = a.find_previous("h2")
    titre = " ".join(h2.get_text().split()).lower() if h2 else ""
    if "en cours" in titre:
        return "en cours"
    if "clos" in titre:
        return "clôturé"
    return "inconnu"

motif = re.compile(
    r"Publié le\s+(?P<pub>\d{1,2}\s+\w+\s+\d{4})\s+"
    r"(?P<ref>[A-Z0-9/\-\u00b0N ]+?)?\s*"
    r"Date limite le\s+(?P<lim>\d{1,2}\s+\w+\s+\d{4})\s+"
    r"(?P<titre>.+)",
    re.IGNORECASE,
)

resultats = []
for a in annonces_html:
    texte = " ".join(a.get_text().split())
    m = motif.search(texte)
    if not m:
        continue
    href = a.get("href")
    lien = href if href.startswith("http") else BASE + href
    reference = (m.group("ref") or "").strip()
    if reference and (len(reference) < 4 or reference.lower() == "n"):
        reference = ""
    resultats.append({
        "cle_unique": lien.rstrip("/").split("/")[-1],
        "reference": reference,
        "intitule": m.group("titre").strip(),
        "date_publication": m.group("pub"),
        "delai_soumission": m.group("lim"),
        "lien": lien,
        "source": "BCEAO",
        "statut_source": section_de(a),
    })

print("Annonces extraites :", len(resultats), "\n")
print(json.dumps(resultats[0], ensure_ascii=False, indent=2))

Annonces extraites : 33 

{
  "cle_unique": "selection-dentreprises-specialisees-pour-la-realisation-dinstallations-solaires-0",
  "reference": "AO/Z00/DPS/008/2026",
  "intitule": "Sélection d’entreprises spécialisées pour la réalisation d’installations solaires photovoltaïques sur les sites de la Banque au Bénin, Burkina, Côte d’Ivoire, Niger et Togo",
  "date_publication": "17 juin 2026",
  "delai_soumission": "31 Juillet 2026",
  "lien": "https://www.bceao.int/fr/appels-offres/selection-dentreprises-specialisees-pour-la-realisation-dinstallations-solaires-0",
  "source": "BCEAO",
  "statut_source": "en cours"
}


## 9. Contrôle de qualité

Je vérifie les valeurs vides par champ, et surtout que la clé unique n'a pas de doublons.

In [32]:
total = len(resultats)
print(f"Total annonces : {total}\n")
print("Valeurs vides par champ :")
for champ in ["cle_unique", "reference", "intitule", "date_publication", "delai_soumission", "lien"]:
    vides = sum(1 for x in resultats if not x.get(champ))
    print(f"  {champ:18} : {vides} vide(s)")
cles = [x["cle_unique"] for x in resultats]
print(f"\nDoublons sur la cle_unique : {len(cles) - len(set(cles))}")
print("Répartition statut_source :", dict(Counter(x["statut_source"] for x in resultats)))

Total annonces : 33

Valeurs vides par champ :
  cle_unique         : 0 vide(s)
  reference          : 15 vide(s)
  intitule           : 0 vide(s)
  date_publication   : 0 vide(s)
  delai_soumission   : 0 vide(s)
  lien               : 0 vide(s)

Doublons sur la cle_unique : 0
Répartition statut_source : {'en cours': 23, 'clôturé': 10}


## 10. Normalisation des dates

Les dates sont en français littéral (`16 juin 2026`). Je les convertis au format ISO `AAAA-MM-JJ`
pour pouvoir les comparer.

In [33]:
MOIS = {
    "janvier": 1, "février": 2, "fevrier": 2, "mars": 3, "avril": 4, "mai": 5,
    "juin": 6, "juillet": 7, "août": 8, "aout": 8, "septembre": 9,
    "octobre": 10, "novembre": 11, "décembre": 12, "decembre": 12,
}

def normaliser_date(txt):
    if not txt:
        return None
    m = re.match(r"(\d{1,2})\s+(\w+)\s+(\d{4})", txt.strip().lower())
    if not m:
        return None
    jour, mois, annee = m.groups()
    num = MOIS.get(mois)
    return f"{annee}-{num:02d}-{int(jour):02d}" if num else None

for x in resultats:
    x["date_publication"] = normaliser_date(x["date_publication"])
    x["delai_soumission"] = normaliser_date(x["delai_soumission"])

print(json.dumps(resultats[0], ensure_ascii=False, indent=2))

{
  "cle_unique": "selection-dentreprises-specialisees-pour-la-realisation-dinstallations-solaires-0",
  "reference": "AO/Z00/DPS/008/2026",
  "intitule": "Sélection d’entreprises spécialisées pour la réalisation d’installations solaires photovoltaïques sur les sites de la Banque au Bénin, Burkina, Côte d’Ivoire, Niger et Togo",
  "date_publication": "2026-06-17",
  "delai_soumission": "2026-07-31",
  "lien": "https://www.bceao.int/fr/appels-offres/selection-dentreprises-specialisees-pour-la-realisation-dinstallations-solaires-0",
  "source": "BCEAO",
  "statut_source": "en cours"
}


## 11. Statut : croiser la section et la date limite

La section du site (`statut_source`) dit si l'annonce est ouverte ou close. C'est l'information
autoritaire : elle attrape même les clôtures anticipées (un prestataire déjà retenu, une annulation).

Mais en complément, je vérifie aussi la **date limite** : si elle est dépassée, l'annonce est de
fait close, même si elle était listée comme « en cours ». Les deux signaux se complètent : la
section pour les clôtures anticipées, la date pour les expirations.

In [34]:
aujourdhui = date.today().isoformat()
print("Date du jour :", aujourdhui, "\n")

for x in resultats:
    # encore ouverte ? (date limite dans le futur ET section "en cours")
    date_ok = (x["delai_soumission"] is not None and x["delai_soumission"] >= aujourdhui)
    x["encore_ouverte"] = bool(date_ok and x["statut_source"] == "en cours")

ouvertes = sum(1 for x in resultats if x["encore_ouverte"])
print(f"Annonces réellement encore ouvertes (section 'en cours' ET date non dépassée) : {ouvertes}")

# cas intéressant : listée "en cours" mais date déjà dépassée
incoherentes = [x for x in resultats
                if x["statut_source"] == "en cours" and not x["encore_ouverte"]]
print(f"Listées 'en cours' mais date limite dépassée : {len(incoherentes)}")

Date du jour : 2026-06-17 

Annonces réellement encore ouvertes (section 'en cours' ET date non dépassée) : 23
Listées 'en cours' mais date limite dépassée : 0


## 12. Vérification : y a-t-il des annonces pertinentes ?

Filtre **très grossier** par mots-clés, juste pour confirmer qu'il y a quelques marchés
informatiques. Le vrai tri sera fait par le scoring (module 2), qui comprend le sens.

In [35]:
mots_cles = ("informati", "numérique", "logiciel", "serveur", "site internet",
             "intranet", "cyber", "réseau", "fibre", "système d'information", "sécurité", "intrusion")
pertinentes = [x for x in resultats if any(m in x["intitule"].lower() for m in mots_cles)]
print(f"{len(pertinentes)} annonce(s) à connotation informatique :\n")
for x in pertinentes:
    print(f"  [{x['statut_source']:9}] {x['intitule'][:80]}")

4 annonce(s) à connotation informatique :

  [en cours ] Report - Sélection d'un prestataire chargé de réaliser la refonte de l'intranet 
  [en cours ] Reprise du câblage fibre optique des immeubles fonctionnels du Siège
  [clôturé  ] Avis d'appel d'offres pour la Sélection d’un prestataire pour les tests d'intrus
  [clôturé  ] Fourniture de trois (03) serveurs physiques X86 et d'une licence ManageEngine Ap


## 13. Sauvegarde

J'enregistre les annonces en JSON, matière première pour l'insertion dans MongoDB.

In [ ]:
with open("../data/bceao_annonces.json", "w", encoding="utf-8") as f:
    json.dump(resultats, f, ensure_ascii=False, indent=2)
print("Sauvegardé :", len(resultats), "annonces dans bceao_annonces.json")

Sauvegardé : 33 annonces dans bceao_annonces.json


## 14. Conclusion et prochaines étapes

**Ce que cette exploration m'a appris :**

- La BCEAO est publique et accessible avec `requests` (pas de navigateur headless), y compris
  pour les annonces closes : elles sont dans le HTML brut.
- Fausse piste corrigée : `find_next_sibling` ne voyait qu'une annonce close, ce qui m'a fait
  croire à tort à du JavaScript. La bonne méthode est `find_previous("h2")` pour rattacher chaque
  annonce à sa section. Leçon : ne pas conclure trop vite à du JavaScript.
- Tout le texte d'une annonce est dans un `<a>`, d'où l'extraction par expression régulière et le
  filtre « Publié le ».
- Le statut se lit de deux façons complémentaires : la **section** du site (`statut_source`, qui
  capte les clôtures anticipées) et la **date limite** (qui capte les expirations).

**Champs collectés :** `cle_unique`, `reference`, `intitule`, `date_publication`,
`delai_soumission`, `lien`, `source`, `statut_source`, `encore_ouverte`.

**Prochaines étapes :**

1. Insérer dans MongoDB avec déduplication sur `cle_unique`.
2. Scorer la pertinence (module 2) ; n'extraire le détail (description, documents PDF) que pour
   les annonces pertinentes (stratégie simple).
3. Pagination : parcourir les pages (`?page=N`).
4. Rafraîchir le statut à chaque collecte (une annonce qui disparaît de « en cours » est close).
5. Automatiser la collecte quotidienne avec n8n.